Example to showcase how to use vLLM + Harmony ReAct Agent + GPT-OSS-20B + MCP-U's RolloutEngine.
You need to first deploy GPT-OSS-20B via vLLM:
```
MODEL=gpt-oss-20b-bf16
API_KEY="token-abc123"
PORT=2024


python -m vllm.entrypoints.openai.api_server \
    --model $MODEL \
    --served-model-name $MODEL \
    --tensor-parallel-size 2 \
    --async-scheduling \
    --gpu-memory-utilization 0.95 \
    --api-key $API_KEY \
    --host "0.0.0.0" \
    --port $PORT \
    --disable-log-requests
```
Then you can go through the following examples.

In [ ]:
import asyncio

from mcpuniverse.rl import RolloutEngine, RolloutConfig

In [ ]:
async def example_basic_harmony():
    """
    Example 1: Basic Harmony ReAct Agent query

    Harmony Agent uses [TOOL_CALL]...[/TOOL_CALL] format for tool calls.
    """
    print("\n" + "=" * 60)
    print("Example 1: Basic Harmony ReAct Agent Query")
    print("=" * 60)

    config = RolloutConfig.from_dict({
        "llm_type": "vllm_local",
        "llm_config": {
            "model_name": "gpt-oss-20b-bf16",
            "base_url": "http://localhost:2024",
            "api_key": "token-abc123",
            "temperature": 0.7,
            "max_completion_tokens": 4096,
        },
        "agent_mode": "harmony",
        "mcp_servers": [{"name": "weather"}],
        "generator": {
            "num_trajectories": 1,
            "max_iterations": 10,
        },
    })

    engine = RolloutEngine(config)

    output = await engine.run([
        {
            "instruction": "What's the weather in New York City?",
            "instance_id": "nyc_weather",
            "evaluators": [
                {"func": "raw", "op": "contain", "value": "Temperature"},
            ],
        }
    ])

    print(f"\nFinal response: {output.responses[0]}")
    print(f"Reward: {output.rewards[0]}")
    print(f"Finish reason: {output.finish_reasons[0]}")

    if output.trajectories:
        traj = output.trajectories[0]
        print(f"\nSteps: {traj['num_steps']}")
        print(f"Tool calls: {traj['num_tool_calls']}")
        print(f"Running time: {traj['running_time']:.2f}s")

        print("\n--- Full trace text (full_trace_text) ---")
        full_text = traj.get("full_trace_text", "")
        print(full_text[:1000])

In [ ]:
from transformers import AutoTokenizer
from mcpuniverse.rl.formatters import get_formatter

async def example_multi_step_reasoning():
    """
    Example 2: Multi-step reasoning

    Harmony Agent decomposes complex problems and calls tools multiple times.
    """
    print("\n" + "=" * 60)
    print("Example 2: Multi-step Reasoning")
    print("=" * 60)

    config = RolloutConfig.from_dict({
        "llm_type": "vllm_local",
        "llm_config": {
            "model_name": "gpt-oss-20b-bf16",
            "base_url": "http://localhost:2024",
            "api_key": "token-abc123",
        },
        "agent_mode": "harmony",
        "mcp_servers": [{"name": "weather"}],
        "generator": {
            "num_trajectories": 1,
            "max_iterations": 15,
        },
    })

    engine = RolloutEngine(config)

    output = await engine.run([
        {
            "instruction": "Compare the weather between NYC and Los Angeles. "
                          "Which city is warmer right now? You need to use the weather tool!!!!",
        }
    ])

    print(f"\nFinal response: {output.responses[0]}")

    if output.trajectories:
        traj = output.trajectories[0]
        print(f"\nTotal steps: {traj['num_steps']}")
        print(f"Tool calls: {traj['num_tool_calls']}")
        print(f"Running time: {traj['running_time']:.2f}s")

        # Tokenize with mask test
        print("\n" + "=" * 60)
        print("Tokenize with Mask Test")
        print("=" * 60)

        tokenizer = AutoTokenizer.from_pretrained(
            "gpt-oss-20b-bf16",
            trust_remote_code=True,
        )

        formatter = get_formatter("gpt_oss")
        full_trace_text = traj.get("full_trace_text", "")

        if full_trace_text:
            formatter_output = formatter.format_trace(
                full_trace_text,
                "Compare the weather between NYC and Los Angeles...",
            )

            prompt_tokens, output_tokens, loss_mask = formatter.tokenize_with_mask(
                formatter_output, tokenizer,
            )

            print(f"\n[Prompt]")
            print(f"  Token count: {len(prompt_tokens)}")
            print(f"  First 20 token IDs: {prompt_tokens[:20]}")
            print(f"  First 20 decoded: {tokenizer.decode(prompt_tokens[:20])}")
            print(f"  Last 20 token IDs: {prompt_tokens[-20:]}")
            print(f"  Last 20 decoded: {tokenizer.decode(prompt_tokens[-20:])}")

            print(f"\n[Output]")
            print(f"  Token count: {len(output_tokens)}")
            print(f"  First 30 token IDs: {output_tokens[:30]}")
            print(f"  First 30 decoded: {tokenizer.decode(output_tokens[:30])}")

            print(f"\n[Loss Mask]")
            print(f"  Total length: {len(loss_mask)}")
            print(f"  Trainable tokens (mask=1): {sum(loss_mask)}")
            print(f"  Non-trainable tokens (mask=0): {len(loss_mask) - sum(loss_mask)}")
            print(f"  First 50 mask values: {loss_mask[:50]}")

            print(f"\n[Segment Details]")
            for i, seg in enumerate(formatter_output.output_segments):
                raw = seg.get("raw", "") or seg.get("content", "")
                trainable = seg.get("trainable", False)
                role = seg.get("role", "")
                channel = seg.get("channel", "")

                if raw:
                    seg_tokens = tokenizer.encode(raw, add_special_tokens=False)
                    raw_preview = raw[:150] + "..." if len(raw) > 150 else raw
                    print(f"\n  Segment {i}: role={role}, channel={channel}")
                    print(f"    Trainable: {'mask=1' if trainable else 'mask=0'}")
                    print(f"    Token count: {len(seg_tokens)}")
                    print(f"    Preview: {raw_preview}")

            # Token-by-token mask visualization
            print("\n" + "=" * 60)
            print("Token-by-Token Mask Visualization")
            print("=" * 60)
            print("Format: [mask] token_text")
            print("  [1] = trainable (model learns)")
            print("  [0] = non-trainable (masked out)")
            print("-" * 60)

            # Find mask transition points
            mask_changes = []
            prev_mask = None
            for idx, mask_val in enumerate(loss_mask):
                if mask_val != prev_mask:
                    mask_changes.append(idx)
                    prev_mask = mask_val

            print(f"\nMask transitions ({len(mask_changes)} total): {mask_changes}")

            # Show tokens around mask boundaries
            print(f"\n[Mask Boundary Details]")
            for change_idx in mask_changes:
                start = max(0, change_idx - 3)
                end = min(len(output_tokens), change_idx + 5)

                print(f"\n  --- Boundary @ token {change_idx} ---")
                for j in range(start, end):
                    token_id = output_tokens[j]
                    mask_val = loss_mask[j]
                    token_text = tokenizer.decode([token_id])
                    token_text_display = token_text.replace("\n", "\\n").replace("\r", "\\r")
                    marker = ">>>" if j == change_idx else "   "
                    print(f"  {marker} [{mask_val}] token[{j}]: '{token_text_display}' (id={token_id})")

            # Show mask for first 100 tokens
            print(f"\n[First 100 Tokens Mask]")
            for j in range(min(100, len(output_tokens))):
                token_id = output_tokens[j]
                mask_val = loss_mask[j]
                token_text = tokenizer.decode([token_id]).replace("\n", "\\n").replace("\r", "\\r")
                if len(token_text) > 20:
                    token_text = token_text[:17] + "..."
                print(f"  [{mask_val}] '{token_text}'", end="")
                if (j + 1) % 5 == 0:
                    print()
            print()

        else:
            print("full_trace_text is empty, cannot test")

In [ ]:
async def example_batch_processing():
    """
    Example 3: Batch processing with evaluators
    """
    print("\n" + "=" * 60)
    print("Example 3: Batch Processing with Evaluators")
    print("=" * 60)

    config = RolloutConfig.from_dict({
        "llm_type": "vllm_local",
        "llm_config": {
            "model_name": "gpt-oss-20b-bf16",
            "base_url": "http://localhost:2024",
            "api_key": "token-abc123",
            "temperature": 0.7,
        },
        "agent_mode": "harmony",
        "use_sample_servers": True,
        "generator": {
            "num_trajectories": 4,
            "max_iterations": 10,
        },
        "dispatcher": {
            "type": "async_pipeline",
            "max_parallel_agents": 32,
        },
    })

    engine = RolloutEngine(config)

    batch = [
        {
            "instruction": "Compare the weather between New York and Los Angeles. Which city is warmer right now?",
            "instance_id": "nyc_vs_la",
            "mcp_servers": [{"name": "weather"}],
            "evaluators": [
                {"func": "raw", "op": "contain", "value": "York"},
                {"func": "raw", "op": "contain", "value": "Angeles"},
            ],
        },
        {
            "instruction": "If current weather in Los Angeles is sunny, find a cafe in Los Angeles. "
                          "If current weather in Los Angeles is rainy, find a park in Los Angeles.",
            "instance_id": "la",
            "mcp_servers": [{"name": "google-maps"}, {"name": "weather"}],
            "evaluators": [
                {"func": "raw", "op": "contain", "value": "Los Angeles"},
            ],
        },
        {
            "instruction": "What's the weather in San Francisco?",
            "instance_id": "sf",
            "mcp_servers": [{"name": "weather"}],
            "evaluators": [
                {"func": "raw", "op": "contain", "value": "Francisco"},
            ],
        },
    ]

    output = await engine.run(batch)

    num_traj = config.generator.num_trajectories
    print(f"\nProcessed {len(batch)} instances, {len(output.responses)} trajectories ({num_traj} per instance)")
    print(f"Success rate: {output.rollout_metrics.get('rollout_metrics/success_rate', 0):.2%}")

    for i, traj in enumerate(output.trajectories):
        instance_id = traj.get("instance_id", i)
        traj_id = traj.get("trajectory_id", 0)
        resp = traj.get("response", "")
        reward = traj.get("reward", 0.0)
        print(f"\nInstance {instance_id} - Trajectory {traj_id}:")
        print(f"  Response: {resp[:200]}..." if len(resp) > 200 else f"  Response: {resp}")
        print(f"  Reward: {reward}")